# Notebook: 00 Pre-Flight Check Script
### Purpose: Test full pipeline to catch issues early


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
# ponytail: PROJECT_ROOT or parent of notebooks/
root = Path(os.getenv("PROJECT_ROOT") or (
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
)).resolve()
sys.path[:0] = [str(root), str(root / "src")]

from src.utils.config import Config
from src.utils.helpers import c, init_notebook, p, simple_estimate_runtime, t
from src.utils.preflight import (
    check_config, check_data, check_dataset, check_disk_space, check_gpu, check_model,
)

config = Config.load(root=root)
init_notebook(config.train.seed)


In [ ]:
t("PRE-FLIGHT CHECK")


In [ ]:
import traceback

checks = [
    ("Configuration", lambda: check_config(config)),
    ("GPU", lambda: check_gpu()),
    ("Data", lambda: check_data(config)),
    ("Dataset", lambda: check_dataset(config)),
    ("Model", lambda: check_model()),
    ("Disk Space", lambda: check_disk_space(config)),
]

results = {}
for name, func in checks:
    try:
        results[name] = func()
        p("")
    except Exception as e:
        results[name] = False
        p(f"✗ {name} check crashed", str(e), color1=c.RED)
        traceback.print_exc()


In [ ]:
try:
    estimate = simple_estimate_runtime(config)
    p("Runtime estimate", estimate)
except Exception as e:
    p("Runtime estimate skipped", str(e), color1=c.ORANGE)


In [ ]:
t("SUMMARY")
passed = sum(1 for v in results.values() if v)
p("Passed", f"{passed}/{len(results)}")
for name, ok in results.items():
    p(name, "OK" if ok else "FAIL", color1=c.GREEN if ok else c.RED)
